In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [18]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

In [42]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()
len(docs)

9

In [43]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_data = splitter.split_documents(docs)
len(splitted_data)

26

In [44]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [45]:
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings
)

In [49]:
query = "Machine Learnings and Data Science Content"
data = vector_store.similarity_search(query=query)

In [13]:
context = ""
for doc in data:
    context += doc.page_content + "\n"



In [14]:
llm = ChatOpenAI(model="gpt-5")

In [17]:
#### Chain - Context_generate | prompt | llm | strparser

In [38]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context += doc.page_content + "\n"

    return {
        "context":context,
        "question":query
    }

In [39]:
prompt = PromptTemplate.from_template("""
    You are a helpful assistant and provide answerd based on the context for user question. and 
    if you don't know the answer, then you can say that 'I dont know.'
    Context: {context}
    Question: {question}
""")

In [24]:
rag_chain = get_context | prompt | llm

In [54]:
res = rag_chain.invoke("What is the value of RBC Count and is it in range ? ")

In [55]:
print(res.content)

RBC Count: 4.47 million/mm³ (×10⁶/µL).  
Is it in range? Yes—within the reference range of 3.80–4.80.
